# Week 75, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w075d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('shopdb.py', 'w', encoding='utf-8') as f:
    f.write('"""A small shop database for practising SQL (Week 75)."""\nimport sqlite3\n\nSCHEMA = """\nCREATE TABLE customers (\n    id     INTEGER PRIMARY KEY,\n    name   TEXT NOT NULL,\n    city   TEXT NOT NULL,\n    joined TEXT NOT NULL,\n    email  TEXT\n);\nCREATE TABLE products (\n    id       INTEGER PRIMARY KEY,\n    name     TEXT NOT NULL,\n    category TEXT NOT NULL,\n    price    REAL NOT NULL,\n    stock    INTEGER\n);\nCREATE TABLE orders (\n    id          INTEGER PRIMARY KEY,\n    customer_id INTEGER NOT NULL REFERENCES customers(id),\n    placed      TEXT NOT NULL,\n    status      TEXT NOT NULL\n);\nCREATE TABLE order_items (\n    order_id   INTEGER NOT NULL REFERENCES orders(id),\n    product_id INTEGER NOT NULL REFERENCES products(id),\n    qty        INTEGER NOT NULL,\n    unit_price REAL NOT NULL,\n    PRIMARY KEY (order_id, product_id)\n);\n"""\n\nDATA = """\nINSERT INTO customers VALUES\n    (1, \'Ada Lovelace\', \'London\', \'2024-01-15\', \'ada@example.com\'),\n    (2, \'Alan Turing\', \'Manchester\', \'2024-03-02\', \'alan@example.com\'),\n    (3, \'Grace Hopper\', \'New York\', \'2024-03-20\', NULL),\n    (4, \'Katherine Johnson\', \'Leeds\', \'2025-02-11\', \'kj@example.com\'),\n    (5, \'Tim Berners-Lee\', \'London\', \'2025-06-30\', NULL),\n    (6, \'Margaret Hamilton\', \'Bristol\', \'2025-09-09\', \'mh@example.com\');\nINSERT INTO products VALUES\n    (1, \'Tea\', \'drinks\', 3.50, 120),\n    (2, \'Coffee\', \'drinks\', 6.00, 80),\n    (3, \'Jam\', \'food\', 4.25, 0),\n    (4, \'Bread\', \'food\', 2.10, 35),\n    (5, \'Mug\', \'kitchen\', 8.00, 15),\n    (6, \'Teapot\', \'kitchen\', 24.00, 4),\n    (7, \'Biscuits\', \'food\', 1.80, 200),\n    (8, \'Hot chocolate\', \'drinks\', 5.50, NULL);\nINSERT INTO orders VALUES\n    (1, 1, \'2025-01-05\', \'delivered\'),\n    (2, 2, \'2025-01-17\', \'delivered\'),\n    (3, 1, \'2025-02-02\', \'cancelled\'),\n    (4, 3, \'2025-03-11\', \'delivered\'),\n    (5, 4, \'2025-03-29\', \'shipped\'),\n    (6, 1, \'2025-04-14\', \'delivered\'),\n    (7, 5, \'2025-05-01\', \'pending\'),\n    (8, 2, \'2025-05-20\', \'delivered\');\nINSERT INTO order_items VALUES\n    (1, 1, 2, 3.50), (1, 7, 3, 1.80),\n    (2, 2, 1, 6.00), (2, 5, 2, 8.00),\n    (3, 6, 1, 24.00),\n    (4, 1, 5, 3.25), (4, 3, 2, 4.25),\n    (5, 4, 1, 2.10), (5, 7, 2, 1.80), (5, 2, 1, 6.00),\n    (6, 6, 1, 22.00), (6, 5, 1, 8.00),\n    (7, 8, 2, 5.50),\n    (8, 1, 1, 3.50), (8, 4, 2, 2.10), (8, 7, 1, 1.80);\n"""\n\n\ndef connect():\n    """A fresh in-memory copy of the shop database."""\n    conn = sqlite3.connect(":memory:")\n    conn.executescript(SCHEMA + DATA)\n    return conn\n\n\ndef rows(sql, params=()):\n    """Run a query on a fresh copy of the database and return every row."""\n    conn = connect()\n    try:\n        return conn.execute(sql, params).fetchall()\n    finally:\n        conn.close()\n\n\ndef show(sql, params=()):\n    """Print a query\'s result as a small table."""\n    conn = connect()\n    cur = conn.execute(sql, params)\n    names = [d[0] for d in cur.description]\n    data = cur.fetchall()\n    widths = [max(len(str(x)) for x in [n] + [r[i] for r in data]) for i, n in enumerate(names)]\n    print("  ".join(n.ljust(w) for n, w in zip(names, widths)).rstrip())\n    for r in data:\n        print("  ".join(str(x).ljust(w) for x, w in zip(r, widths)).rstrip())\n    conn.close()\n')

**Predict before you run:** what will this print?



In [ ]:
from shopdb import rows
print(rows("SELECT count(*), count(email), count(DISTINCT city) FROM customers"))

**Predict before you run:** what will this print?



In [ ]:
from shopdb import rows
print(rows("""
SELECT c.name, count(o.id) FROM customers c LEFT JOIN orders o ON o.customer_id = c.id
GROUP BY c.id HAVING count(o.id) = 0
"""))

**Predict before you run:** what will this print?



In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE t (x INTEGER CHECK (x > 0))")
try:
    with conn:
        conn.execute("INSERT INTO t VALUES (1)")
        conn.execute("INSERT INTO t VALUES (-1)")
except sqlite3.IntegrityError:
    pass
print(conn.execute("SELECT count(*) FROM t").fetchone())

**Predict before you run:** what will this print?



In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE t (a, b, c)")
conn.execute("CREATE INDEX t_ab ON t (a, b)")
for where in ("a = 1", "b = 1", "a = 1 AND b > 2"):
    detail = conn.execute(f"EXPLAIN QUERY PLAN SELECT c FROM t WHERE {where}").fetchone()[-1]
    print(where, "->", "index" if "INDEX" in detail else "scan")

**Predict before you run:** what will this print?



In [ ]:
from shopdb import rows
print(rows("SELECT 1 IN (1, NULL), 2 IN (1, NULL), 2 NOT IN (1, NULL)"))

### Exercise: Customers who bought both

Write `BOTH`: the names of customers who have bought **both** Tea and Biscuits (in orders that weren't cancelled), alphabetically. Then `EVERY_CATEGORY`: the names of customers who have bought from **every** product category (in orders that weren't cancelled), alphabetically, without writing the category names into the query.

In [ ]:
BOTH = ___
EVERY_CATEGORY = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_both():\n    \"grouping with HAVING over sets\"\n    from shopdb import rows\n    assert rows(BOTH) == [(\"Ada Lovelace\",), (\"Alan Turing\",)]\n    assert rows(EVERY_CATEGORY) == [(\"Ada Lovelace\",), (\"Alan Turing\",)]\n    assert \"drinks\" not in EVERY_CATEGORY and \"kitchen\" not in EVERY_CATEGORY")

### Exercise: Insert or update

Write `count_word(conn, word)`: add one to the count for `word` (lower-cased) in a table `words (word TEXT PRIMARY KEY, n INTEGER NOT NULL)`, inserting it with `n = 1` if it isn't there, in **one** statement using `INSERT ... ON CONFLICT (word) DO UPDATE SET n = n + 1`; commit. Then `top(conn, k)`: the `k` most common `(word, n)` pairs, ties alphabetical.

In [ ]:
import sqlite3

def count_word(conn, word):
    ___

def top(conn, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_upsert():\n    \"an upsert\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    conn.execute(\"CREATE TABLE words (word TEXT PRIMARY KEY, n INTEGER NOT NULL)\")\n    for w in \"the cat and The hat and the bat\".split():\n        count_word(conn, w)\n    assert top(conn, 3) == [(\"the\", 3), (\"and\", 2), (\"bat\", 1)] and not conn.in_transaction\n    assert \"ON CONFLICT\" in source.upper()")

### Exercise: Forge: a library's loans

Write `MIGRATIONS`, `LoanError` and `Library` as described above. Use parameters for every value.

In [ ]:
import sqlite3
from datetime import date, timedelta

MIGRATIONS = [
    ___
]

class LoanError(Exception):
    pass

class Library:
    def __init__(self, conn):
        self.conn = conn
        conn.execute("PRAGMA foreign_keys = ON")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def make():\n    import sqlite3\n    lib = Library(sqlite3.connect(\":memory:\"))\n    lib.setup()\n    emma = lib.add_book(\"978-1\", \"Emma\", \"Austen\", copies=2)\n    dune = lib.add_book(\"978-2\", \"Dune\", \"Herbert\")\n    ada = lib.add_member(\"Ada\", \"Ada@Example.com\")\n    bob = lib.add_member(\"Bob\", \"bob@example.com\")\n    return lib, emma, dune, ada, bob\n\ndef test_setup():\n    \"migrations, constraints and indexes\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    lib = Library(conn)\n    assert lib.setup() == [1, 2] and lib.setup() == []\n    assert conn.execute(\"PRAGMA user_version\").fetchone() == (2,)\n    names = {r[0] for r in conn.execute(\"SELECT name FROM sqlite_master WHERE type = 'index'\")}\n    assert {\"loans_book_id\", \"loans_member_id\"} <= names\n    lib.add_book(\"978-1\", \"Emma\", \"Austen\")\n    for bad in [lambda: lib.add_book(\"978-1\", \"Again\", \"X\"), lambda: lib.add_book(\"978-9\", \"Neg\", \"X\", copies=-1)]:\n        try:\n            bad()\n            assert False\n        except sqlite3.IntegrityError:\n            pass\n    assert conn.execute(\"PRAGMA foreign_keys\").fetchone() == (1,)\n\ndef test_lending():\n    \"the lending rules\"\n    from datetime import date\n    lib, emma, dune, ada, bob = make()\n    day = date(2026, 11, 2)\n    first = lib.lend(emma, ada, day)\n    lib.lend(emma, bob, day)\n    for book, member, message in [(emma, ada, \"no copies available\"), (99, ada, \"no such book\"), (dune, 99, \"no such member\")]:\n        try:\n            lib.lend(book, member, day)\n            assert False\n        except LoanError as e:\n            assert str(e) == message\n    assert lib.give_back(first, date(2026, 11, 16)) == 0\n    assert lib.lend(emma, ada, day) and lib.lend(dune, ada, day)\n    third = lib.add_book(\"978-3\", \"Middlemarch\", \"Eliot\")\n    lib.lend(third, ada, day)\n    fourth = lib.add_book(\"978-4\", \"Ulysses\", \"Joyce\")\n    try:\n        lib.lend(fourth, ada, day)\n        assert False\n    except LoanError as e:\n        assert str(e) == \"member has too many loans\"\n    try:\n        lib.lend(fourth, bob, date(2026, 11, 30))\n        assert False\n    except LoanError as e:\n        assert str(e) == \"member has overdue books\"\n\ndef test_returns_and_reports():\n    \"fines, overdue lists and popularity\"\n    from datetime import date\n    lib, emma, dune, ada, bob = make()\n    l1 = lib.lend(emma, ada, date(2026, 10, 1))\n    l2 = lib.lend(dune, bob, date(2026, 10, 20))\n    l3 = lib.lend(emma, bob, date(2026, 10, 10))\n    assert lib.overdue(date(2026, 11, 5)) == [(\"Ada\", \"Emma\", 21), (\"Bob\", \"Emma\", 12), (\"Bob\", \"Dune\", 2)]\n    assert lib.give_back(l1, date(2026, 10, 18)) == 30 and lib.give_back(l3, date(2027, 6, 1)) == 500\n    try:\n        lib.give_back(l1, date(2026, 10, 19))\n        assert False\n    except LoanError as e:\n        assert str(e) == \"already returned\"\n    lib.lend(emma, ada, date(2026, 11, 1))\n    assert lib.popular(2) == [(\"Emma\", 3), (\"Dune\", 1)]\n    assert lib.summary(bob, date(2026, 11, 5)) == {\"total\": 2, \"active\": 1, \"overdue\": 1}\n    assert lib.summary(ada, date(2026, 11, 5)) == {\"total\": 2, \"active\": 1, \"overdue\": 0}")

> **Where the rules live.** Look at where each rule is enforced. Some live in the **schema** (unique ISBNs and emails, non-negative copies, loans that must point at real books and members): the database refuses to break them, whatever code is calling. Others live in **`lend`** (free copies, the loan limit, no overdue books), and hold only because every loan goes through that method, inside one transaction. A second program writing to the same database could bypass them. That's the usual trade-off: put the rules that must never break into constraints where you can, and keep the rest behind one well-tested door.

### Exercise: Forge bonus: a database health check

Write `audit(conn)`: return a list of problems with a SQLite database, in this order: `"integrity: MESSAGE"` for each line of `PRAGMA integrity_check` other than `ok`; `"broken key: TABLE row ROWID -> PARENT"` for each row `PRAGMA foreign_key_check` reports (it works even while foreign keys are off); `"no primary key: TABLE"` for each table (other than SQLite's own `sqlite_` tables) where no column of `PRAGMA table_info` is part of the primary key; and `"unindexed foreign key: TABLE.COLUMN"` for each single-column foreign key (from `PRAGMA foreign_key_list(TABLE)`) whose column isn't the **first** column of any index on the table (from `PRAGMA index_list` and `PRAGMA index_info`), or of the table's own multi-column primary key. List tables in alphabetical order.

In [ ]:
import sqlite3

def audit(conn):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_audit():\n    \"PRAGMAs that find problems\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    conn.executescript(\"\"\"\n    CREATE TABLE authors (id INTEGER PRIMARY KEY, name TEXT);\n    CREATE TABLE books (id INTEGER PRIMARY KEY, author_id INTEGER REFERENCES authors(id), title TEXT);\n    CREATE TABLE reviews (book_id INTEGER REFERENCES books(id), text TEXT);\n    CREATE TABLE tags (book_id INTEGER REFERENCES books(id), tag TEXT, PRIMARY KEY (book_id, tag));\n    CREATE TABLE loans (id INTEGER PRIMARY KEY, book_id INTEGER REFERENCES books(id));\n    CREATE INDEX loans_book ON loans (book_id);\n    INSERT INTO authors VALUES (1, 'Austen');\n    INSERT INTO books VALUES (1, 1, 'Emma'), (2, 7, 'Lost');\n    INSERT INTO reviews VALUES (9, 'no such book');\n    \"\"\")\n    assert audit(conn) == [\n        \"broken key: books row 2 -> authors\",\n        \"broken key: reviews row 1 -> books\",\n        \"no primary key: reviews\",\n        \"unindexed foreign key: books.author_id\",\n        \"unindexed foreign key: reviews.book_id\",\n    ]\n    clean = sqlite3.connect(\":memory:\")\n    clean.executescript(\"CREATE TABLE a (id INTEGER PRIMARY KEY); CREATE TABLE b (id INTEGER PRIMARY KEY, a_id INTEGER REFERENCES a(id)); CREATE INDEX b_a ON b (a_id);\")\n    assert audit(clean) == []")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Forge: a library's loans

Write the data layer for a small lending library: a `Library(conn)` class that turns on foreign keys for its connection, with these methods (`LoanError` is an exception class you define):

- **`setup()`** applies the numbered migrations in `MIGRATIONS` (as in Day 3's challenge: each in its own explicit transaction, recording the version in `PRAGMA user_version`) and returns the numbers applied. Migration 1 creates `books (id, isbn TEXT NOT NULL UNIQUE, title TEXT NOT NULL, author TEXT NOT NULL, copies INTEGER NOT NULL CHECK (copies >= 0))`, `members (id, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE)` and `loans (id, book_id, member_id, out, due, returned)`, with the two ids as foreign keys, the dates as ISO text, and `returned` NULL until the book comes back. Migration 2 creates the indexes `loans_book_id` and `loans_member_id`.
- **`add_book(isbn, title, author, copies=1)`** and **`add_member(name, email)`** (storing the email lower-cased) insert and return the new id.
- **`lend(book_id, member_id, today)`** (with `today` a `date`) checks, in this order, inside one transaction: that the book exists (`"no such book"`), the member exists (`"no such member"`), a copy is free (books on loan are those with `returned IS NULL`; `"no copies available"`), the member has no overdue loans (unreturned, with `due` before today; `"member has overdue books"`) and fewer than 3 active loans (`"member has too many loans"`); raising `LoanError` with that message on the first failure. Otherwise it records a loan due 14 days later and returns its id.
- **`give_back(loan_id, today)`** marks the loan returned (`LoanError("no such loan")` or `LoanError("already returned")`) and returns the fine in pence: 10 per day late, at most 500.
- **`overdue(today)`** lists `(member name, title, days late)` for unreturned overdue loans, most days late first, then by name and title.
- **`popular(limit=3)`** gives `(title, number of loans ever)` for the most-borrowed books (only books that have been borrowed), most first, ties by title.
- **`summary(member_id, today)`** returns `{"total": ..., "active": ..., "overdue": ...}` counts of the member's loans.

## Forge bonus: a database health check

## Weekly quiz

## Week 75 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can write SELECT queries with WHERE, ORDER BY, LIMIT, CASE and DISTINCT, and reason about NULL. ([Day 1](w075d1.html))
- I can join tables (inner and left), group and aggregate with HAVING, and use subqueries and WITH, avoiding fan-out. ([Day 2](w075d2.html))
- I can use sqlite3 with parameters, transactions, dict rows, type conversion and foreign keys, and write migrations. ([Day 3](w075d3.html))
- I can explain what an ORM does, build mapping, identity maps and query objects, and avoid N+1 queries. ([Day 4](w075d4.html))
- I can prevent lost updates, use savepoints, read query plans, and choose composite and covering indexes. ([Day 5](w075d5.html))
- I built a library's data layer and a database health check. (Today)

Tomorrow is a **rest day**. Next week is **networking and HTTP**: how the web works, the HTTP protocol in detail, calling web APIs, sockets, and writing an HTTP server of your own.